<a href="https://colab.research.google.com/github/arasskazov/gost-kb/blob/main/utils/GOST_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ RAG-пайплайн для базы знаний ГОСТ Р 77.*

Этот ноутбук строит локальную RAG-систему поверх репозитория [arasskazov/gost-kb](https://github.com/arasskazov/gost-kb).

**Что делает:**
- Загружает все `full.md` из GitHub
- Строит векторный индекс (ChromaDB + многоязычные эмбеддинги)
- Отвечает на вопросы через DeepSeek API

**Порядок запуска:** выполняйте ячейки сверху вниз (Shift+Enter).

---
**Время первого запуска:** ~5–10 минут (загрузка модели эмбеддингов ~1.2 GB)  
**Последующие запуски:** ~1–2 минуты (если сессия Colab не перезапускалась)


## Шаг 1 — Установка зависимостей

In [ ]:
# %%capture
!pip install llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface
!pip install chromadb sentence-transformers openai
print('✅ Зависимости установлены')

✅ Зависимости установлены


## Шаг 2 — Введите ваш DeepSeek API ключ

In [ ]:
# Шаг 2 — задаём ключ напрямую (без getpass)
DEEPSEEK_API_KEY = "sk-fb4ae2bb4e574cb9aded98a2a0dc86ad"
print("✅ Ключ сохранён")

✅ Ключ сохранён


## Шаг 3 — Загрузка всех стандартов из GitHub

In [ ]:
import requests
from llama_index.core import Document

STANDARDS = [
    '77-001', '77-002', '77-101', '77-102',
    '77-301', '77-302', '77-303', '77-304',
    '77-305', '77-306', '77-402', '77-403', '77-404'
]
BASE_URL = 'https://raw.githubusercontent.com/arasskazov/gost-kb/main'

documents = []
failed = []

for std in STANDARDS:
    # Загружаем full.md
    url_full = f'{BASE_URL}/standards/lci/{std}/full.md'
    r = requests.get(url_full, timeout=15)
    if r.status_code == 200 and len(r.text) > 100:
        documents.append(Document(
            text=r.text,
            metadata={'standard': std, 'source': url_full, 'type': 'full'}
        ))
        print(f'  ✅ {std}: {len(r.text):,} символов')
    else:
        failed.append(std)
        print(f'  ❌ {std}: не загружен (статус {r.status_code})')

    # Загружаем summary.md как дополнительный документ
    url_sum = f'{BASE_URL}/standards/lci/{std}/summary.md'
    r2 = requests.get(url_sum, timeout=15)
    if r2.status_code == 200 and len(r2.text) > 50:
        documents.append(Document(
            text=r2.text,
            metadata={'standard': std, 'source': url_sum, 'type': 'summary'}
        ))

print(f'\n📚 Загружено документов: {len(documents)}')
if failed:
    print(f'⚠️  Не загружены: {failed}')

  ✅ 77-001: 12,343 символов
  ✅ 77-002: 51,991 символов
  ✅ 77-101: 15,602 символов
  ✅ 77-102: 22,889 символов
  ✅ 77-301: 21,663 символов
  ✅ 77-302: 57,062 символов
  ✅ 77-303: 45,942 символов
  ✅ 77-304: 39,617 символов
  ✅ 77-305: 27,826 символов
  ✅ 77-306: 75,092 символов
  ✅ 77-402: 27,598 символов
  ✅ 77-403: 37,658 символов
  ✅ 77-404: 33,423 символов

📚 Загружено документов: 26


## Шаг 4 — Построение векторного индекса

> ⏳ Первый запуск займёт 5–8 минут: скачивается модель эмбеддингов ~1.2 GB

In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print('🔄 Загружаем модель эмбеддингов (multilingual-e5-large)...')
embed_model = HuggingFaceEmbedding(
    model_name='intfloat/multilingual-e5-large',
    max_length=512
)
Settings.embed_model = embed_model
print('✅ Модель загружена')

# ChromaDB в памяти (достаточно для работы в Colab)
chroma_client = chromadb.EphemeralClient()
chroma_collection = chroma_client.get_or_create_collection(
    'gost_standards',
    metadata={'hnsw:space': 'cosine'}
)
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print('🔄 Индексируем документы...')
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True
)
print(f'\n✅ Индекс построен: {len(documents)} документов')

🔄 Загружаем модель эмбеддингов (multilingual-e5-large)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

✅ Модель загружена
🔄 Индексируем документы...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/237 [00:00<?, ?it/s]


✅ Индекс построен: 26 документов


In [ ]:
!pip install llama_index llama_index.llms.openai_like
import llama_index
import llama_index.llms.openai_like

## Шаг 5 — Настройка DeepSeek и поискового движка

In [ ]:
from llama_index.llms.openai_like import OpenAILike
from llama_index.core import Settings

SYSTEM_PROMPT = """Ты — эксперт-нормировщик по стандартам серии ГОСТ Р 77.* \
(управление жизненным циклом изделий).

Правила:
1. Отвечай ТОЛЬКО на основе предоставленных фрагментов стандартов.
2. Всегда указывай номер ГОСТ и раздел/пункт (например: ГОСТ Р 77.001, п. 3.4).
3. Если информация в предоставленных фрагментах отсутствует — так и скажи,
   не придумывай нормы.
4. При противоречиях между стандартами — укажи оба варианта с источниками.
5. Используй точные формулировки из текста стандарта."""

llm = OpenAILike(
    model='deepseek-chat',
    api_base='https://api.deepseek.com',
    api_key=DEEPSEEK_API_KEY,
    temperature=0.1,
    max_tokens=2048,
    is_chat_model=True,
    system_prompt=SYSTEM_PROMPT
)
Settings.llm = llm

# Поисковый движок: топ-6 релевантных чанков
query_engine = index.as_query_engine(
    similarity_top_k=6,
    response_mode='compact'
)
print('✅ Поисковый движок готов')

✅ Поисковый движок готов


## Шаг 6 — Задаём вопросы!

Измените текст в переменной `question` и запустите ячейку.

In [ ]:
def ask(question: str, show_sources: bool = True):
    print(f'\n❓ Вопрос: {question}')
    print('─' * 60)
    response = query_engine.query(question)
    print(f'💬 Ответ:\n{response}')
    if show_sources:
        print('\n📎 Источники:')
        seen = set()
        for node in response.source_nodes:
            std = node.metadata.get('standard', '?')
            src_type = node.metadata.get('type', '?')
            score = node.score or 0
            key = f'{std}/{src_type}'
            if key not in seen:
                print(f'  → {std} ({src_type}), релевантность: {score:.3f}')
                seen.add(key)
    return response

# ── Пример 1: базовый вопрос ──────────────────────────────────────────────
ask('Что такое жизненный цикл изделия и какие стадии он включает?')


❓ Вопрос: Что такое жизненный цикл изделия и какие стадии он включает?
────────────────────────────────────────────────────────────
💬 Ответ:
Жизненный цикл (ЖЦ) изделия представляет собой совокупность последовательных стадий, каждая из которых характеризуется определенной целью и набором выполняемых работ. Состав стадий жизненного цикла, а также виды и содержание выполняемых работ для отдельных видов (типов) изделий могут изменяться (уточняться) по согласованию с заказчиком (представительством заказчика). В составе стадий ЖЦ для контроля результатов работ выделяют этапы, характеризующиеся целями работ, планируемыми результатами, методами контроля выполненных работ или полученных характеристик изделия. Для моментов окончания этапов в модели ЖЦ устанавливают контрольные рубежи (КР), на которых предполагается принятие решений о продолжении работ, их прекращении или о переходе к следующему этапу (ГОСТ Р 77.102, п. 5.4, 5.5).

📎 Источники:
  → 77-102 (full), релевантность: 0.865
  → 77-002

Response(response='Жизненный цикл (ЖЦ) изделия представляет собой совокупность последовательных стадий, каждая из которых характеризуется определенной целью и набором выполняемых работ. Состав стадий жизненного цикла, а также виды и содержание выполняемых работ для отдельных видов (типов) изделий могут изменяться (уточняться) по согласованию с заказчиком (представительством заказчика). В составе стадий ЖЦ для контроля результатов работ выделяют этапы, характеризующиеся целями работ, планируемыми результатами, методами контроля выполненных работ или полученных характеристик изделия. Для моментов окончания этапов в модели ЖЦ устанавливают контрольные рубежи (КР), на которых предполагается принятие решений о продолжении работ, их прекращении или о переходе к следующему этапу (ГОСТ Р 77.102, п. 5.4, 5.5).', source_nodes=[NodeWithScore(node=TextNode(id_='7f46fa50-0b56-4c4c-81db-a5f56adef57d', embedding=None, metadata={'standard': '77-102', 'source': 'https://raw.githubusercontent.com/arassk

In [ ]:
# ── Пример 2: поиск по конкретному стандарту ─────────────────────────────
ask('Какова область применения ГОСТ 77.102?')

In [ ]:
# ── Пример 3: сравнительный запрос ───────────────────────────────────────
ask('В чём разница между понятиями "изделие" и "продукт" в стандартах серии 77?')

In [ ]:
# ── Ваш вопрос ───────────────────────────────────────────────────────────
question = 'Введите ваш вопрос здесь'
ask(question)

## Шаг 7 — Интерактивный режим (чат)

Запустите ячейку и вводите вопросы в поле ввода.  
Введите `выход` для остановки.

In [ ]:
print('🤖 ГОСТ-эксперт готов. Задавайте вопросы (введите "выход" для остановки)\n')
while True:
    q = input('Вопрос: ').strip()
    if not q or q.lower() in ('выход', 'exit', 'quit'):
        print('Сессия завершена.')
        break
    ask(q)

🤖 ГОСТ-эксперт готов. Задавайте вопросы (введите "выход" для остановки)


❓ Вопрос: В чём разница между понятиями "изделие" и "продукт" в стандартах серии 77?
────────────────────────────────────────────────────────────
💬 Ответ:
В соответствии с предоставленными фрагментами стандартов серии ГОСТ Р 77.*, различие между понятиями «изделие» и «продукт» не установлено. В тексте стандарта (п. 3.1.25) указано, что для целей настоящего стандарта «изделие» рассматривается совместно с комплексом связанных с ним объектов. Термин «продукт» в данном контексте не упоминается, и его соотношение с понятием «изделие» не раскрывается.

📎 Источники:
  → 77-002 (full), релевантность: 0.871
  → 77-302 (full), релевантность: 0.869
  → 77-303 (full), релевантность: 0.865


---
## 📝 Советы

**Хорошие вопросы для этой базы:**
- *Что означает термин X по ГОСТ 77.001?*
- *Какие требования к документации на стадии разработки?*
- *Какие стандарты серии 77 регулируют вопросы утилизации?*
- *Перечислите ключевые термины из ГОСТ 77.301*

**Если сессия Colab перезапустилась:**  
Нужно снова выполнить все ячейки (Runtime → Run all).  
Индекс пересоздастся за ~5 минут.

**Обновление базы (при появлении новых стандартов):**  
Добавьте номер нового стандарта в список `STANDARDS` в Шаге 3 и перезапустите с Шага 3.

**Стоимость DeepSeek API:**  
deepseek-chat стоит $0.27/1M токенов (вход) и $1.10/1M (выход).  
Один типичный запрос к этой базе ≈ $0.001–0.003.
